# 01 — Data Exploration

Load the observed data (40 replicates) and visualise the three data sources:
- Infected fraction timeseries
- Rewiring count timeseries
- Final degree histograms

Also compute summary statistics and run simulator sanity checks.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_all
from summary_stats import compute_observed_summaries, SUMMARY_NAMES

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 4)

In [ ]:
inf_ts, rew_ts, deg_hist = load_all()
print(f'Replicates: {inf_ts.shape[0]}, Time steps: {inf_ts.shape[1]-1}')
print(f'Degree bins: {deg_hist.shape[1]}')

## 1.1 Infected Fraction Timeseries

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
t = np.arange(inf_ts.shape[1])
for r in range(inf_ts.shape[0]):
    ax.plot(t, inf_ts[r], color='steelblue', alpha=0.25, linewidth=0.8)
ax.plot(t, np.median(inf_ts, axis=0), color='darkred', linewidth=2, label='Median')
ax.fill_between(t, np.percentile(inf_ts, 25, axis=0), np.percentile(inf_ts, 75, axis=0),
                color='steelblue', alpha=0.2, label='IQR')
ax.set_xlabel('Time step')
ax.set_ylabel('Infected fraction')
ax.set_title('Observed Infected Fraction Timeseries (40 replicates)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/obs_infected_ts.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.2 Rewiring Count Timeseries

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for r in range(rew_ts.shape[0]):
    ax.plot(t, rew_ts[r], color='seagreen', alpha=0.25, linewidth=0.8)
ax.plot(t, np.median(rew_ts, axis=0), color='darkred', linewidth=2, label='Median')
ax.fill_between(t, np.percentile(rew_ts, 25, axis=0), np.percentile(rew_ts, 75, axis=0),
                color='seagreen', alpha=0.2, label='IQR')
ax.set_xlabel('Time step')
ax.set_ylabel('Rewire count')
ax.set_title('Observed Rewiring Timeseries (40 replicates)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/obs_rewiring_ts.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.3 Final Degree Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
degrees = np.arange(31)
mean_hist = np.mean(deg_hist, axis=0)
std_hist = np.std(deg_hist, axis=0)
ax.bar(degrees, mean_hist, yerr=std_hist, color='coral', alpha=0.7, capsize=2)
ax.set_xlabel('Degree')
ax.set_ylabel('Count (mean +/- std)')
ax.set_title('Final Degree Histogram (averaged over 40 replicates)')
ax.set_xticks(degrees[::2])
plt.tight_layout()
plt.savefig('../figures/obs_degree_hist.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 Summary Statistics

In [ ]:
s_obs, s_per_rep = compute_observed_summaries(inf_ts, rew_ts, deg_hist)

print('Observed summary statistics (median over 40 replicates):')
print('-' * 55)
for name, val in zip(SUMMARY_NAMES, s_obs):
    print(f'  {name:35s}: {val:.4f}')

fig, axes = plt.subplots(4, 3, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i < 12:
        ax.hist(s_per_rep[:, i], bins=15, color='steelblue', alpha=0.7, edgecolor='white')
        ax.axvline(s_obs[i], color='red', linestyle='--', linewidth=1.5)
        ax.set_title(SUMMARY_NAMES[i], fontsize=9)
    else:
        ax.set_visible(False)
plt.suptitle('Distribution of Summary Statistics Across 40 Replicates', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/obs_summary_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 Simulator Sanity Check

Run the simulator at several hand-picked parameter values to build intuition about the model behaviour and compare against the observed data.

In [ ]:
from simulator import simulate_fast

# Warm up Numba JIT
_ = simulate_fast(0.2, 0.1, 0.3, seed=0)

params_to_try = [
    (0.15, 0.05, 0.3, 'Low beta, low gamma, moderate rho'),
    (0.25, 0.08, 0.5, 'Moderate beta/gamma, high rho'),
    (0.35, 0.10, 0.6, 'High beta, moderate gamma, high rho'),
    (0.20, 0.15, 0.2, 'Moderate beta, high gamma, low rho'),
]

fig, axes = plt.subplots(len(params_to_try), 3, figsize=(15, 3.5 * len(params_to_try)))

for row, (beta, gamma, rho, label) in enumerate(params_to_try):
    for rep in range(5):
        inf, rew, deg = simulate_fast(beta, gamma, rho, seed=row * 100 + rep)
        axes[row, 0].plot(inf, alpha=0.5)
        axes[row, 1].plot(rew, alpha=0.5)
        if rep == 0:
            axes[row, 2].bar(np.arange(31), deg, alpha=0.5)

    # Overlay observed median
    axes[row, 0].plot(np.median(inf_ts, axis=0), 'k--', lw=1.5, label='Observed median')
    axes[row, 1].plot(np.median(rew_ts, axis=0), 'k--', lw=1.5, label='Observed median')
    axes[row, 2].bar(np.arange(31), np.mean(deg_hist, axis=0), alpha=0.3, color='red', label='Observed mean')

    axes[row, 0].set_ylabel(f'b={beta}, g={gamma}\nr={rho}', fontsize=9)
    if row == 0:
        axes[row, 0].set_title('Infected fraction')
        axes[row, 1].set_title('Rewiring counts')
        axes[row, 2].set_title('Degree histogram')
        axes[row, 0].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../figures/simulator_sanity_check.png', dpi=150, bbox_inches='tight')
plt.show()